# Task 18: DeepLoc 2.0 context-first experiment

This task uses DeepLoc 2.0 in stages. The primary experiment adds **WT protein context** to the fixed 64D XGBoost model. WT–MT sorting-signal deltas are evaluated only if a sensitivity pilot shows a meaningful dynamic range. Localisation deltas are retained as an exploratory diagnostic and are not the primary feature set.

Primary ablations on Task 15's fixed gene-disjoint folds:

1. 64D fixed baseline;
2. 64D + 9 human-relevant WT localisation probabilities (exclude plastid);
3. 64D + 6 WT sorting-signal probabilities (SP, MT, NLS, NES, PTS, GPI; exclude plant signals and TM);
4. 64D + both WT feature groups;
5. sensitivity analysis adding WT TM probability, which may duplicate existing TMD features.

The local DeepLoc package is documented in `models/models.md`. Use one model setting throughout; the current setting is Fast/ESM1b.

In [2]:
from pathlib import Path
import hashlib
import re
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

ROOT = Path("/mnt/volume6/czj/labLGN/LabLZ")
BASE = ROOT / "xgboost_trial"
MODEL_PACKAGE = ROOT / "models/deeploc2_package"
SOURCE_CSV = ROOT / "data_preparation/cell2024_model_single_subst.csv"
FEATURES_CSV = BASE / "features_v3.csv"
TASK15_OOF = BASE / "task15_full64_oof.csv"
TMD_CSV = BASE / "tmd_features.csv"
DEEPlOC_DIR = MODEL_PACKAGE / "deeploc2"
FASTA_DIR = DEEPlOC_DIR / "fasta"
MANIFEST_CSV = DEEPlOC_DIR / "sequence_manifest.csv"
PAIR_MAP_CSV = DEEPlOC_DIR / "variant_pair_map.csv"
WT_PREDICTIONS_CSV = DEEPlOC_DIR / "wt_predictions_fast_esm1b.csv"
PILOT_PREDICTIONS_CSV = DEEPlOC_DIR / "pilot_mt_predictions_fast_esm1b.csv"
FULL_PREDICTIONS_CSV = DEEPlOC_DIR / "all_predictions_fast_esm1b.csv"
CONTEXT_FEATURES_CSV = BASE / "deeploc_wt_context_features.csv"
SENSITIVITY_CSV = BASE / "deeploc_sorting_delta_sensitivity.csv"
OOF_CSV = BASE / "task18_deeploc_context_oof.csv"
METRICS_CSV = BASE / "task18_deeploc_context_metrics.csv"
IMPORTANCE_CSV = BASE / "task18_deeploc_context_importance.csv"
BOOTSTRAP_CSV = BASE / "task18_deeploc_context_bootstrap.csv"

DEEPlOC_DIR.mkdir(parents=True, exist_ok=True)
FASTA_DIR.mkdir(parents=True, exist_ok=True)

LOCATIONS = ["cytoplasm", "nucleus", "extracellular", "cell_membrane", "mitochondrion", "plastid", "endoplasmic_reticulum", "lysosome_vacuole", "golgi_apparatus", "peroxisome"]
SIGNALS = ["signal_peptide", "transmembrane_domain", "mitochondrial_transit_peptide", "chloroplast_transit_peptide", "thylakoid_transit_peptide", "nuclear_localisation_signal", "nuclear_export_signal", "peroxisomal_targeting_signal", "gpi_anchor"]
HUMAN_LOCATIONS = [c for c in LOCATIONS if c != "plastid"]
CORE_SIGNALS = ["signal_peptide", "mitochondrial_transit_peptide", "nuclear_localisation_signal", "nuclear_export_signal", "peroxisomal_targeting_signal", "gpi_anchor"]
TM_SIGNAL = ["transmembrane_domain"]
STRUCT_COLS = ["plddt", "sasa", "rsa", "ss_helix", "ss_strand", "ss_coil", "delta_hydrophobicity"]
DDG_COLS = ["ddg_esm2", "ddg_struct", "ddg_rasp", "ddg_foldx"]
TMD_COLS = ["in_TMD", "dist_to_nearest_TMD", "delta_membrane_insertion"]
RANDOM_STATE = 42
N_COMPONENTS = 50
FASTA_BATCH_SIZE = 500
MODEL_SETTING = "Fast/ESM1b"

## 18.1 Build WT-first manifests and FASTA files

WT sequences are deduplicated independently, so the context experiment can run before any MT inference. A deterministic pilot enriches positives, TMD variants, and terminal variants to test whether the sorting-signal head responds to single substitutions. The pilot is diagnostic and must not be used as a performance cohort.

In [4]:
source = pd.read_csv(SOURCE_CSV)
required = ["Gene", "Mutation_used", "sequence", "mutant_sequence", "Mislocalized"]
missing = [c for c in required if c not in source.columns]
assert not missing, f"Missing source columns: {missing}"
source["KEY"] = source["Gene"].astype(str) + "||" + source["Mutation_used"].astype(str)
assert len(source) == 2179 and source["KEY"].is_unique
assert source["Mislocalized"].isin([0, 1]).all() and int(source["Mislocalized"].sum()) == 236

def clean_sequence(value):
    if not isinstance(value, str):
        return None
    sequence = re.sub(r"\s+", "", value).upper()
    return sequence if sequence and re.fullmatch(r"[ACDEFGHIKLMNPQRSTVWYU]+", sequence) else None

def mutation_position(value):
    match = re.fullmatch(r"[A-Z](\d+)[A-Z]", str(value))
    return int(match.group(1)) if match else np.nan

source["wt_sequence"] = source["sequence"].map(clean_sequence)
source["mt_sequence"] = source["mutant_sequence"].map(clean_sequence)
assert source[["wt_sequence", "mt_sequence"]].notna().all().all(), "Invalid or missing amino-acid sequence"
source["mutation_position"] = source["Mutation_used"].map(mutation_position)
source["sequence_length"] = source["wt_sequence"].str.len()

all_sequences = pd.concat([source["wt_sequence"], source["mt_sequence"]], ignore_index=True).drop_duplicates().tolist()
sequence_to_id = {sequence: f"DL{index:05d}" for index, sequence in enumerate(all_sequences, start=1)}
manifest = pd.DataFrame({"sequence_id": [sequence_to_id[s] for s in all_sequences], "sequence": all_sequences})
manifest["original_length"] = manifest["sequence"].str.len()
manifest["fast_clipped"] = manifest["original_length"] > 1022
manifest["effective_length_fast"] = manifest["original_length"].clip(upper=1022)
manifest["sha256"] = manifest["sequence"].map(lambda s: hashlib.sha256(s.encode()).hexdigest())
assert manifest["sequence_id"].is_unique and manifest["sequence"].is_unique

pairs = source[["KEY", "Gene", "Mutation_used", "Mislocalized", "mutation_position", "sequence_length"]].copy()
pairs["wt_sequence_id"] = source["wt_sequence"].map(sequence_to_id)
pairs["mt_sequence_id"] = source["mt_sequence"].map(sequence_to_id)
pairs["wt_equals_mt"] = pairs["wt_sequence_id"] == pairs["mt_sequence_id"]
pairs["mutation_retained_fast"] = (pairs["sequence_length"] <= 1022) | (pairs["mutation_position"] <= 511) | (pairs["mutation_position"] > pairs["sequence_length"] - 511)

tmd = pd.read_csv(TMD_CSV, usecols=["KEY", "in_TMD"])
pairs = pairs.merge(tmd, on="KEY", how="left", validate="one_to_one")
terminal = (pairs["mutation_position"] <= 50) | (pairs["mutation_position"] > pairs["sequence_length"] - 50)
pairs["pilot_variant"] = pairs["Mislocalized"].eq(1) | pairs["in_TMD"].eq(1) | terminal

manifest["is_wt"] = manifest["sequence_id"].isin(pairs["wt_sequence_id"])
manifest["is_pilot_mt"] = manifest["sequence_id"].isin(pairs.loc[pairs["pilot_variant"], "mt_sequence_id"])
manifest["is_remaining_mt"] = ~manifest["is_wt"] & ~manifest["is_pilot_mt"]
manifest.to_csv(MANIFEST_CSV, index=False)
pairs.to_csv(PAIR_MAP_CSV, index=False)

def write_fasta_batches(table, prefix):
    paths = []
    for start in range(0, len(table), FASTA_BATCH_SIZE):
        path = FASTA_DIR / f"{prefix}_{start // FASTA_BATCH_SIZE + 1:02d}.fasta"
        with path.open("w") as handle:
            for row in table.iloc[start:start + FASTA_BATCH_SIZE].itertuples(index=False):
                handle.write(f">{row.sequence_id}\n{row.sequence}\n")
        paths.append(path)
    return paths

wt_manifest = manifest.loc[manifest["is_wt"]].copy()
pilot_mt_manifest = manifest.loc[manifest["is_pilot_mt"] & ~manifest["is_wt"]].copy()
remaining_mt_manifest = manifest.loc[manifest["is_remaining_mt"]].copy()
wt_fastas = write_fasta_batches(wt_manifest, "wt")
pilot_fastas = write_fasta_batches(pilot_mt_manifest, "pilot_mt_nonwt")
remaining_fastas = write_fasta_batches(remaining_mt_manifest, "remaining_mt")
print(f"Variants: {len(pairs)}; unique WT: {len(wt_manifest)}; pilot MT (non-WT): {len(pilot_mt_manifest)}; remaining MT: {len(remaining_mt_manifest)}")
print(f"WT FASTA batches: {len(wt_fastas)}; pilot non-WT FASTA batches: {len(pilot_fastas)}; remaining MT FASTA batches: {len(remaining_fastas)}")
print(f"Fast clipping affects {manifest['fast_clipped'].sum()} unique sequences and removes the mutation for {(~pairs['mutation_retained_fast']).sum()} variants")
print(f"Pilot variants: {pairs['pilot_variant'].sum()}; remaining variants: {(~pairs['pilot_variant']).sum()}")

Variants: 2179; unique WT: 871; pilot MT (non-WT): 762; remaining MT: 1386
WT FASTA batches: 2; pilot non-WT FASTA batches: 2; remaining MT FASTA batches: 3
Fast clipping affects 108 unique sequences and removes the mutation for 7 variants
Pilot variants: 770; remaining variants: 1409


## 18.2 Local inference adapter

The downloaded CLI discards sorting-signal probabilities when writing CSV. This adapter calls the packaged Fast/ESM1b ensemble and preserves its internal 9D signal output. It does not alter package source code. Run WT inference first. Run pilot MT inference only for the sensitivity diagnostic.

The ninth internal signal dimension is treated as GPI-anchor according to the model's nine thresholds and the published class order. The assertion below fails if the package output dimensions differ.

In [3]:
# Set to True only in the environment where the licensed package and ESM1b backbone are available.
RUN_LOCAL_INFERENCE = True
DEVICE = "cuda:1"  # use "cpu" or "mps" only after a successful one-sequence smoke test
TOKENS_PER_BATCH = 8192

REMAINING_MT_PREDICTIONS_CSV = DEEPlOC_DIR / "remaining_mt_predictions_fast_esm1b.csv"

def run_fast_deeploc(table, output_csv, device=DEVICE, tokens_per_batch=TOKENS_PER_BATCH):
    import torch
    from esm import Alphabet

    package_root = str(MODEL_PACKAGE.resolve())
    if package_root not in sys.path:
        sys.path.insert(0, package_root)
    from DeepLoc2.data import BatchConverter, FastaBatchedDatasetTorch
    from DeepLoc2.model import ESM1bE2E

    work = table[["sequence_id", "sequence", "original_length", "fast_clipped", "effective_length_fast"]].copy()
    work["sequence"] = work["sequence"].map(lambda s: s if len(s) <= 1022 else s[:511] + s[-511:])
    model = ESM1bE2E().to(device).eval()
    alphabet = Alphabet.from_architecture("roberta_large")
    dataset_df = work.rename(columns={"sequence_id": "ACC", "sequence": "Sequence"}).reset_index(drop=True)
    dataset = FastaBatchedDatasetTorch(dataset_df)
    batches = dataset.get_batch_indices(tokens_per_batch, extra_toks_per_seq=2)
    loader = torch.utils.data.DataLoader(dataset, collate_fn=BatchConverter(alphabet), batch_sampler=batches)
    records = []
    with torch.inference_mode():
        for tokens, lengths, mask, identifiers in loader:
            localisation, _, signal = model(tokens, lengths, mask)
            assert localisation.shape == (len(identifiers), 11) and signal.shape == (len(identifiers), 9)
            for i, sequence_id in enumerate(identifiers):
                record = {"sequence_id": sequence_id, "model_setting": MODEL_SETTING}
                record.update({name: float(localisation[i, j + 1]) for j, name in enumerate(LOCATIONS)})
                record.update({name: float(signal[i, j]) for j, name in enumerate(SIGNALS)})
                records.append(record)
    result = work.drop(columns="sequence").merge(pd.DataFrame(records), on="sequence_id", validate="one_to_one")
    assert len(result) == len(work) and result[LOCATIONS + SIGNALS].applymap(np.isfinite).all().all()
    assert ((result[LOCATIONS + SIGNALS] >= 0) & (result[LOCATIONS + SIGNALS] <= 1)).all().all()
    result.to_csv(output_csv, index=False)
    return result

if RUN_LOCAL_INFERENCE:
    wt_predictions = run_fast_deeploc(wt_manifest, WT_PREDICTIONS_CSV)
    pilot_predictions = run_fast_deeploc(pilot_mt_manifest, PILOT_PREDICTIONS_CSV) if len(pilot_mt_manifest) else pd.DataFrame()
    print(f"Saved WT predictions: {WT_PREDICTIONS_CSV}")
    print(f"Saved pilot MT predictions: {PILOT_PREDICTIONS_CSV}")
    # Full MT inference: remaining non-pilot, non-WT sequences
    remaining_predictions = run_fast_deeploc(remaining_mt_manifest, REMAINING_MT_PREDICTIONS_CSV) if len(remaining_mt_manifest) else pd.DataFrame()
    print(f"Saved remaining MT predictions: {REMAINING_MT_PREDICTIONS_CSV}")
    # Combine all MT predictions (pilot + remaining)
    mt_parts = [p for p in [pilot_predictions, remaining_predictions] if len(p)]
    all_mt_predictions = pd.concat(mt_parts, ignore_index=True) if mt_parts else pd.DataFrame()
    if len(all_mt_predictions):
        all_mt_predictions.to_csv(FULL_PREDICTIONS_CSV, index=False)
    print(f"Saved all MT predictions ({len(all_mt_predictions)} sequences): {FULL_PREDICTIONS_CSV}")
else:
    print("Inference disabled. Set RUN_LOCAL_INFERENCE=True on the licensed DeepLoc environment.")

Lightning automatically upgraded your loaded checkpoint from v1.5.8 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../models/deeploc2_package/DeepLoc2/models/models_esm1b/0_1Layer.ckpt`
Lightning automatically upgraded your loaded checkpoint from v1.5.8 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../models/deeploc2_package/DeepLoc2/models/models_esm1b/1_1Layer.ckpt`
Lightning automatically upgraded your loaded checkpoint from v1.5.8 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../models/deeploc2_package/DeepLoc2/models/models_esm1b/2_1Layer.ckpt`
Lightning automatically upgraded your loaded checkpoint from v1.5.8 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../models/deeploc2_package/Deep

Saved WT predictions: /mnt/volume6/czj/labLGN/LabLZ/models/deeploc2_package/deeploc2/wt_predictions_fast_esm1b.csv
Saved pilot MT predictions: /mnt/volume6/czj/labLGN/LabLZ/models/deeploc2_package/deeploc2/pilot_mt_predictions_fast_esm1b.csv


Lightning automatically upgraded your loaded checkpoint from v1.5.8 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../models/deeploc2_package/DeepLoc2/models/models_esm1b/0_1Layer.ckpt`
Lightning automatically upgraded your loaded checkpoint from v1.5.8 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../models/deeploc2_package/DeepLoc2/models/models_esm1b/1_1Layer.ckpt`
Lightning automatically upgraded your loaded checkpoint from v1.5.8 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../models/deeploc2_package/DeepLoc2/models/models_esm1b/2_1Layer.ckpt`
Lightning automatically upgraded your loaded checkpoint from v1.5.8 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../models/deeploc2_package/Deep

Saved remaining MT predictions: /mnt/volume6/czj/labLGN/LabLZ/models/deeploc2_package/deeploc2/remaining_mt_predictions_fast_esm1b.csv
Saved all MT predictions (2148 sequences): /mnt/volume6/czj/labLGN/LabLZ/models/deeploc2_package/deeploc2/all_predictions_fast_esm1b.csv


## 18.3 Validate WT predictions and create context features

Plant-specific outputs are retained in the raw file for auditing and excluded from the primary human feature sets. WT TM is separated because the 64D model already contains three TMD features.

In [5]:
if not WT_PREDICTIONS_CSV.exists():
    raise FileNotFoundError(f"Run WT DeepLoc inference first: {WT_PREDICTIONS_CSV}")
wt_pred = pd.read_csv(WT_PREDICTIONS_CSV)
required_pred = ["sequence_id", "model_setting"] + LOCATIONS + SIGNALS
assert not [c for c in required_pred if c not in wt_pred.columns]
assert wt_pred["sequence_id"].is_unique and set(wt_pred["sequence_id"]) == set(wt_manifest["sequence_id"])
assert wt_pred["model_setting"].eq(MODEL_SETTING).all()
assert ((wt_pred[LOCATIONS + SIGNALS] >= 0) & (wt_pred[LOCATIONS + SIGNALS] <= 1)).all().all()

context = pairs[["KEY", "Gene", "wt_sequence_id", "mutation_retained_fast"]].merge(wt_pred, left_on="wt_sequence_id", right_on="sequence_id", validate="many_to_one")
rename = {c: f"deeploc_wt_loc_{c}" for c in LOCATIONS} | {c: f"deeploc_wt_signal_{c}" for c in SIGNALS}
context = context.rename(columns=rename)
WT_LOC_COLS = [f"deeploc_wt_loc_{c}" for c in HUMAN_LOCATIONS]
WT_SIGNAL_COLS = [f"deeploc_wt_signal_{c}" for c in CORE_SIGNALS]
WT_TM_COL = ["deeploc_wt_signal_transmembrane_domain"]
save_cols = ["KEY", "Gene", "wt_sequence_id", "mutation_retained_fast"] + list(rename.values())
context[save_cols].to_csv(CONTEXT_FEATURES_CSV, index=False)
assert len(context) == 2179 and context["KEY"].is_unique
print(context[WT_LOC_COLS + WT_SIGNAL_COLS + WT_TM_COL].describe().T.to_string())
print(f"Saved: {CONTEXT_FEATURES_CSV}")

                                                  count      mean       std       min       25%       50%       75%       max
deeploc_wt_loc_cytoplasm                         2179.0  0.363774  0.229438  0.063596  0.170955  0.262338  0.595741  0.891974
deeploc_wt_loc_nucleus                           2179.0  0.301608  0.250416  0.024460  0.108796  0.178887  0.438141  0.975841
deeploc_wt_loc_extracellular                     2179.0  0.222233  0.299371  0.001577  0.032400  0.080076  0.230231  0.972043
deeploc_wt_loc_cell_membrane                     2179.0  0.351644  0.277580  0.011282  0.131615  0.241430  0.554190  0.947008
deeploc_wt_loc_mitochondrion                     2179.0  0.223970  0.245703  0.013783  0.076016  0.134285  0.237283  0.968101
deeploc_wt_loc_endoplasmic_reticulum             2179.0  0.256573  0.200074  0.008299  0.112720  0.210298  0.333920  0.938517
deeploc_wt_loc_lysosome_vacuole                  2179.0  0.267719  0.181133  0.004585  0.109124  0.235202  0.394753  0

## 18.4 Sorting-signal sensitivity pilot

Sensitivity is described by the distribution of signed and absolute MT−WT changes, threshold exceedance rates, and Fast-clipping retention. Exact zero counts alone are not used because exported precision can create artificial zeros. Pairwise correlation with `ddg_esm2` is descriptive; continuation is decided from dynamic range, mechanism-relevant strata, and later paired OOF ablation.

In [6]:
if PILOT_PREDICTIONS_CSV.exists():
    pilot_nonwt = pd.read_csv(PILOT_PREDICTIONS_CSV)
    combined_pred = pd.concat([wt_pred, pilot_nonwt], ignore_index=True).drop_duplicates("sequence_id", keep="last")
    pilot_pairs = pairs.loc[pairs["pilot_variant"]].copy()
    wt_values = combined_pred.rename(columns={"sequence_id": "wt_sequence_id", **{c: f"wt_{c}" for c in LOCATIONS + SIGNALS}})
    mt_values = combined_pred.rename(columns={"sequence_id": "mt_sequence_id", **{c: f"mt_{c}" for c in LOCATIONS + SIGNALS}})
    pilot = pilot_pairs.merge(wt_values[["wt_sequence_id"] + [f"wt_{c}" for c in LOCATIONS + SIGNALS]], on="wt_sequence_id", validate="many_to_one")
    pilot = pilot.merge(mt_values[["mt_sequence_id"] + [f"mt_{c}" for c in LOCATIONS + SIGNALS]], on="mt_sequence_id", validate="many_to_one")
    for name in LOCATIONS + SIGNALS:
        pilot[f"delta_{name}"] = pilot[f"mt_{name}"] - pilot[f"wt_{name}"]
    rows = []
    for name in SIGNALS:
        values = pilot[f"delta_{name}"].to_numpy()
        abs_values = np.abs(values)
        rows.append({"signal": name, "n": len(values), "median_delta": np.median(values), "median_abs_delta": np.median(abs_values), "p90_abs_delta": np.quantile(abs_values, 0.90), "p95_abs_delta": np.quantile(abs_values, 0.95), "p99_abs_delta": np.quantile(abs_values, 0.99), "fraction_abs_gt_0.001": np.mean(abs_values > 0.001), "fraction_abs_gt_0.01": np.mean(abs_values > 0.01), "fraction_abs_gt_0.05": np.mean(abs_values > 0.05)})
    sensitivity = pd.DataFrame(rows)
    sensitivity.to_csv(SENSITIVITY_CSV, index=False)
    print(sensitivity.to_string(index=False))
    print(f"Pilot variants whose mutation is removed by Fast clipping: {(~pilot['mutation_retained_fast']).sum()}")
else:
    pilot = None
    print("Pilot MT predictions are not available. WT-context modelling can continue; sorting-delta modelling remains gated.")

                       signal   n  median_delta  median_abs_delta  p90_abs_delta  p95_abs_delta  p99_abs_delta  fraction_abs_gt_0.001  fraction_abs_gt_0.01  fraction_abs_gt_0.05
               signal_peptide 770  8.595292e-06          0.000872       0.008753       0.014432       0.037654               0.472727              0.080519              0.003896
         transmembrane_domain 770  6.146729e-08          0.000570       0.006391       0.010034       0.023306               0.405195              0.050649              0.000000
mitochondrial_transit_peptide 770 -8.555944e-07          0.000086       0.003427       0.006781       0.033299               0.176623              0.033766              0.003896
  chloroplast_transit_peptide 770  7.006747e-07          0.000020       0.001159       0.003084       0.016352               0.110390              0.020779              0.001299
    thylakoid_transit_peptide 770 -2.356755e-07          0.000012       0.000144       0.000284       0.001817

## 18.5 Fixed-fold WT-context ablation

All imputation, scaling, and PCA are fitted on training-fold data only. The 64D predictions are reused from Task 15. New models use the same fold assignment and XGBoost settings. Primary comparisons are evaluated with pooled OOF AUROC/AUPRC, precision at recall 0.20 and 0.40, and paired gene-cluster bootstrap.

In [7]:
df = pd.read_csv(FEATURES_CSV)
task15 = pd.read_csv(TASK15_OOF)
assert len(df) == 2179 and df["KEY"].is_unique
df = df.merge(task15[["KEY", "fold", "oof_stability_tmd_64", "final_alphamissense_score"]], on="KEY", how="left", validate="one_to_one")
for name in DDG_COLS:
    table = pd.read_csv(BASE / f"{name}.csv")
    df = df.merge(table[["KEY", name]], on="KEY", how="left", validate="one_to_one")
tmd_full = pd.read_csv(TMD_CSV)
df = df.merge(tmd_full[["KEY"] + TMD_COLS], on="KEY", how="left", validate="one_to_one")
df = df.merge(context[["KEY"] + WT_LOC_COLS + WT_SIGNAL_COLS + WT_TM_COL], on="KEY", how="left", validate="one_to_one")

esm_cols = [c for c in df.columns if c.startswith("esm_")]
assert len(esm_cols) == 1280 and int(df["Mislocalized"].sum()) == 236
y = df["Mislocalized"].astype(int).to_numpy()
fold_id = df["fold"].astype(int).to_numpy()
X_esm = df[esm_cols].to_numpy(np.float32)
X_struct = df[STRUCT_COLS].to_numpy(np.float32)
X_base_extra = df[DDG_COLS + TMD_COLS].to_numpy(np.float32)
oof = {"xgboost_64": df["oof_stability_tmd_64"].to_numpy(np.float32)}
feature_sets = {
    "wt_loc_73": WT_LOC_COLS,
    "wt_signal_70": WT_SIGNAL_COLS,
    "wt_context_79": WT_LOC_COLS + WT_SIGNAL_COLS,
    "wt_context_tm_80": WT_LOC_COLS + WT_SIGNAL_COLS + WT_TM_COL,
}
importance_parts = []

for model_name, added_cols in feature_sets.items():
    oof[model_name] = np.full(len(df), np.nan, dtype=np.float32)
    X_added = df[added_cols].to_numpy(np.float32)
    for fold in sorted(np.unique(fold_id)):
        train_idx = np.flatnonzero(fold_id != fold)
        test_idx = np.flatnonzero(fold_id == fold)
        esm_imp, esm_scaler = SimpleImputer(strategy="median"), StandardScaler()
        esm_train = esm_scaler.fit_transform(esm_imp.fit_transform(X_esm[train_idx]))
        esm_test = esm_scaler.transform(esm_imp.transform(X_esm[test_idx]))
        pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
        pc_train, pc_test = pca.fit_transform(esm_train), pca.transform(esm_test)
        struct_imp, struct_scaler = SimpleImputer(strategy="median"), StandardScaler()
        struct_train = struct_scaler.fit_transform(struct_imp.fit_transform(X_struct[train_idx]))
        struct_test = struct_scaler.transform(struct_imp.transform(X_struct[test_idx]))
        extra = np.hstack([X_base_extra, X_added])
        extra_imp = SimpleImputer(strategy="median")
        extra_train, extra_test = extra_imp.fit_transform(extra[train_idx]), extra_imp.transform(extra[test_idx])
        X_train = np.hstack([pc_train, struct_train, extra_train]).astype(np.float32)
        X_test = np.hstack([pc_test, struct_test, extra_test]).astype(np.float32)
        model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.5, objective="binary:logistic", eval_metric="aucpr", random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist")
        model.fit(X_train, y[train_idx], sample_weight=compute_sample_weight("balanced", y[train_idx]), verbose=False)
        oof[model_name][test_idx] = model.predict_proba(X_test)[:, 1]
        names = [f"PC{i + 1}" for i in range(N_COMPONENTS)] + STRUCT_COLS + DDG_COLS + TMD_COLS + added_cols
        importance_parts.append(pd.DataFrame({"model": model_name, "fold": fold, "feature": names, "importance": model.feature_importances_}))
    assert np.isfinite(oof[model_name]).all()

def precision_at_recall(y_true, score, target):
    precision, recall, _ = precision_recall_curve(y_true, score)
    order = np.argsort(recall)
    return float(np.interp(target, recall[order], precision[order]))

def metric_row(scope, model_name, truth, score):
    return {"scope": scope, "model": model_name, "n": len(truth), "positives": int(truth.sum()), "auroc": roc_auc_score(truth, score), "auprc": average_precision_score(truth, score), "precision_at_recall_0.20": precision_at_recall(truth, score, 0.20), "precision_at_recall_0.40": precision_at_recall(truth, score, 0.40)}

records = [metric_row("full_oof", name, y, score) for name, score in oof.items()]
paired_am = df["final_alphamissense_score"].notna().to_numpy()
records.append(metric_row("paired_alphamissense", "alphamissense", y[paired_am], df.loc[paired_am, "final_alphamissense_score"].to_numpy()))
records.extend(metric_row("paired_alphamissense", name, y[paired_am], score[paired_am]) for name, score in oof.items())
metrics_df = pd.DataFrame(records)
print(metrics_df.to_string(index=False))

out = df[["KEY", "Gene", "Mutation_used", "Mislocalized", "fold", "final_alphamissense_score"]].copy()
for name, score in oof.items():
    out[f"oof_{name}"] = score
out.to_csv(OOF_CSV, index=False)
metrics_df.to_csv(METRICS_CSV, index=False)
importance = pd.concat(importance_parts).groupby(["model", "feature"], as_index=False)["importance"].mean()
importance["rank_within_model"] = importance.groupby("model")["importance"].rank(method="min", ascending=False).astype(int)
importance.sort_values(["model", "rank_within_model"]).to_csv(IMPORTANCE_CSV, index=False)

               scope            model    n  positives    auroc    auprc  precision_at_recall_0.20  precision_at_recall_0.40
            full_oof       xgboost_64 2179        236 0.642168 0.198098                  0.266061                  0.186119
            full_oof        wt_loc_73 2179        236 0.632043 0.224672                  0.358796                  0.213284
            full_oof     wt_signal_70 2179        236 0.656036 0.247897                  0.313681                  0.212524
            full_oof    wt_context_79 2179        236 0.652422 0.242465                  0.335704                  0.222326
            full_oof wt_context_tm_80 2179        236 0.655683 0.243483                  0.385616                  0.219228
paired_alphamissense    alphamissense 2140        235 0.649071 0.161931                  0.178030                  0.174074
paired_alphamissense       xgboost_64 2140        235 0.644242 0.199925                  0.265537                  0.186508
paired_a

## 18.6 Paired gene-cluster bootstrap

This resamples genes from the fixed OOF predictions. It estimates evaluation-sample uncertainty conditional on the trained predictions; it does not include fold, PCA, or retraining uncertainty.

In [8]:
rng = np.random.default_rng(RANDOM_STATE)
unique_genes = df["Gene"].astype(str).unique()
gene_to_idx = {gene: np.flatnonzero(df["Gene"].astype(str).to_numpy() == gene) for gene in unique_genes}
bootstrap_rows = []
for replicate in range(2000):
    sampled_genes = rng.choice(unique_genes, size=len(unique_genes), replace=True)
    idx = np.concatenate([gene_to_idx[gene] for gene in sampled_genes])
    if np.unique(y[idx]).size < 2:
        continue
    base_auc = roc_auc_score(y[idx], oof["xgboost_64"][idx])
    base_ap = average_precision_score(y[idx], oof["xgboost_64"][idx])
    for name in feature_sets:
        bootstrap_rows.append({"replicate": replicate, "comparison": f"{name}-xgboost_64", "delta_auroc": roc_auc_score(y[idx], oof[name][idx]) - base_auc, "delta_auprc": average_precision_score(y[idx], oof[name][idx]) - base_ap})
bootstrap = pd.DataFrame(bootstrap_rows)
bootstrap.to_csv(BOOTSTRAP_CSV, index=False)
summary = bootstrap.groupby("comparison").agg(delta_auroc_mean=("delta_auroc", "mean"), delta_auroc_low=("delta_auroc", lambda x: np.quantile(x, 0.025)), delta_auroc_high=("delta_auroc", lambda x: np.quantile(x, 0.975)), delta_auprc_mean=("delta_auprc", "mean"), delta_auprc_low=("delta_auprc", lambda x: np.quantile(x, 0.025)), delta_auprc_high=("delta_auprc", lambda x: np.quantile(x, 0.975))).reset_index()
print(summary.to_string(index=False))
print("Do not claim superiority when the paired interval crosses zero.")

                 comparison  delta_auroc_mean  delta_auroc_low  delta_auroc_high  delta_auprc_mean  delta_auprc_low  delta_auprc_high
   wt_context_79-xgboost_64          0.009691        -0.028863          0.046555          0.044589         0.007440          0.085855
wt_context_tm_80-xgboost_64          0.013155        -0.026806          0.050360          0.044249         0.008217          0.085210
       wt_loc_73-xgboost_64         -0.010710        -0.048598          0.023213          0.026701        -0.007455          0.065250
    wt_signal_70-xgboost_64          0.013756        -0.019438          0.044861          0.047155         0.013017          0.084284
Do not claim superiority when the paired interval crosses zero.


## 18.7 Full WT–MT sorting-signal deltas

The sensitivity pilot confirmed non-trivial tails for SP, NLS, and NES. Build 7 sorting-signal delta features (6 core signals + TM) from the full MT predictions. Localisation deltas remain exploratory.

In [9]:
DELTA_CSV = BASE / "deeploc_sorting_delta_features.csv"

if FULL_PREDICTIONS_CSV.exists():
    all_pred = pd.read_csv(FULL_PREDICTIONS_CSV)
    # Combine WT + all MT predictions into a single lookup
    wt_lookup = wt_pred.copy()
    mt_lookup = all_pred.copy()
    combined_pred = pd.concat([wt_lookup, mt_lookup], ignore_index=True).drop_duplicates("sequence_id", keep="last")

    # Build per-variant delta features
    wt_vals = combined_pred.rename(columns={"sequence_id": "wt_sequence_id",
        **{c: f"wt_{c}" for c in LOCATIONS + SIGNALS}})
    mt_vals = combined_pred.rename(columns={"sequence_id": "mt_sequence_id",
        **{c: f"mt_{c}" for c in LOCATIONS + SIGNALS}})

    delta = pairs[["KEY", "Gene", "wt_sequence_id", "mt_sequence_id", "mutation_retained_fast"]].copy()
    delta = delta.merge(wt_vals[["wt_sequence_id"] + [f"wt_{c}" for c in LOCATIONS + SIGNALS]],
        on="wt_sequence_id", validate="many_to_one")
    delta = delta.merge(mt_vals[["mt_sequence_id"] + [f"mt_{c}" for c in LOCATIONS + SIGNALS]],
        on="mt_sequence_id", validate="many_to_one")

    # 7 sorting-signal deltas: 6 core + TM
    SORTING_DELTA_SIGNALS = CORE_SIGNALS + ["transmembrane_domain"]
    SORTING_DELTA_COLS = [f"deeploc_delta_{c}" for c in SORTING_DELTA_SIGNALS]
    for name in SORTING_DELTA_SIGNALS:
        delta[f"deeploc_delta_{name}"] = delta[f"mt_{name}"] - delta[f"wt_{name}"]

    # Localisation deltas (exploratory only)
    LOC_DELTA_COLS = [f"deeploc_delta_loc_{c}" for c in HUMAN_LOCATIONS]
    for name in HUMAN_LOCATIONS:
        delta[f"deeploc_delta_loc_{name}"] = delta[f"mt_{name}"] - delta[f"wt_{name}"]

    delta[["KEY"] + SORTING_DELTA_COLS + LOC_DELTA_COLS].to_csv(DELTA_CSV, index=False)
    assert len(delta) == 2179 and delta["KEY"].is_unique

    print(f"Sorting-signal delta features ({len(SORTING_DELTA_COLS)}):")
    print(delta[SORTING_DELTA_COLS].describe().T.to_string())
    print(f"\nSaved: {DELTA_CSV}")
    print(f"Variants with missing MT prediction: {delta[SORTING_DELTA_COLS].isna().any(axis=1).sum()}")
else:
    delta = None
    print(f"Full MT predictions not found: {FULL_PREDICTIONS_CSV}. Run cell 5 first.")

Sorting-signal delta features (7):
                                              count      mean       std       min       25%       50%       75%       max
deeploc_delta_signal_peptide                 2179.0  0.000124  0.005570 -0.070732 -0.000346  0.000003  0.000431  0.054087
deeploc_delta_mitochondrial_transit_peptide  2179.0  0.000101  0.004691 -0.037917 -0.000083 -0.000002  0.000043  0.078214
deeploc_delta_nuclear_localisation_signal    2179.0 -0.000338  0.007788 -0.104960 -0.000346 -0.000001  0.000225  0.105045
deeploc_delta_nuclear_export_signal          2179.0 -0.000614  0.009505 -0.181746 -0.000375 -0.000002  0.000049  0.169176
deeploc_delta_peroxisomal_targeting_signal   2179.0 -0.000231  0.006694 -0.106511 -0.000065 -0.000001  0.000053  0.129895
deeploc_delta_gpi_anchor                     2179.0  0.000113  0.002872 -0.074109 -0.000023  0.000004  0.000068  0.051552
deeploc_delta_transmembrane_domain           2179.0  0.000025  0.003903 -0.047211 -0.000202  0.000000  0.000328

## 18.8 Fixed-fold ablation with sorting-signal deltas

Four-model comparison on Task 15's fixed folds: (1) 64D baseline, (2) wt_signal_70, (3) 64D + 7 sorting-signal deltas, (4) wt_signal_70 + 7 sorting-signal deltas.

In [10]:
DELTA_OOF_CSV = BASE / "task18_deeploc_delta_oof.csv"
DELTA_METRICS_CSV = BASE / "task18_deeploc_delta_metrics.csv"
DELTA_IMPORTANCE_CSV = BASE / "task18_deeploc_delta_importance.csv"
DELTA_BOOTSTRAP_CSV = BASE / "task18_deeploc_delta_bootstrap.csv"

if delta is not None:
    delta_df = df.merge(delta[["KEY"] + SORTING_DELTA_COLS], on="KEY", how="left", validate="one_to_one")

    delta_oof = {}
    delta_feature_sets = {
        "xgboost_64":               [],
        "wt_signal_70":             WT_SIGNAL_COLS,
        "xgboost_64_plus_delta_71":  SORTING_DELTA_COLS,
        "wt_signal_70_plus_delta_77": WT_SIGNAL_COLS + SORTING_DELTA_COLS,
    }
    delta_importance_parts = []

    for model_name, added_cols in delta_feature_sets.items():
        if model_name == "xgboost_64":
            delta_oof[model_name] = df["oof_stability_tmd_64"].to_numpy(np.float32)
            continue
        if model_name == "wt_signal_70":
            delta_oof[model_name] = oof["wt_signal_70"]
            continue

        delta_oof[model_name] = np.full(len(df), np.nan, dtype=np.float32)
        X_added = delta_df[added_cols].to_numpy(np.float32)
        for fold in sorted(np.unique(fold_id)):
            train_idx = np.flatnonzero(fold_id != fold)
            test_idx = np.flatnonzero(fold_id == fold)
            esm_imp, esm_scaler = SimpleImputer(strategy="median"), StandardScaler()
            esm_train = esm_scaler.fit_transform(esm_imp.fit_transform(X_esm[train_idx]))
            esm_test = esm_scaler.transform(esm_imp.transform(X_esm[test_idx]))
            pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
            pc_train, pc_test = pca.fit_transform(esm_train), pca.transform(esm_test)
            struct_imp, struct_scaler = SimpleImputer(strategy="median"), StandardScaler()
            struct_train = struct_scaler.fit_transform(struct_imp.fit_transform(X_struct[train_idx]))
            struct_test = struct_scaler.transform(struct_imp.transform(X_struct[test_idx]))
            # All models: X_base_extra + added_cols only
            # added_cols already contains WT_SIGNAL_COLS for wt_signal_70_plus_delta_77
            extra = np.hstack([X_base_extra, X_added])
            extra_imp = SimpleImputer(strategy="median")
            extra_train, extra_test = extra_imp.fit_transform(extra[train_idx]), extra_imp.transform(extra[test_idx])
            X_train = np.hstack([pc_train, struct_train, extra_train]).astype(np.float32)
            X_test = np.hstack([pc_test, struct_test, extra_test]).astype(np.float32)
            model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8,
                colsample_bytree=0.5, objective="binary:logistic", eval_metric="aucpr",
                random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist")
            model.fit(X_train, y[train_idx],
                sample_weight=compute_sample_weight("balanced", y[train_idx]), verbose=False)
            delta_oof[model_name][test_idx] = model.predict_proba(X_test)[:, 1]
            names = [f"PC{i + 1}" for i in range(N_COMPONENTS)] + STRUCT_COLS + DDG_COLS + TMD_COLS + added_cols
            # Hard dimension checks
            expected_dim = 64 + len(added_cols)
            assert X_train.shape[1] == expected_dim, f"{model_name}: expected {expected_dim}D, got {X_train.shape[1]}D"
            assert len(names) == expected_dim, f"{model_name}: expected {expected_dim} names, got {len(names)}"
            assert len(names) == len(set(names)), f"{model_name}: duplicate feature names detected"
            delta_importance_parts.append(pd.DataFrame({"model": model_name, "fold": fold, "feature": names,
                "importance": model.feature_importances_}))
        assert np.isfinite(delta_oof[model_name]).all()

    delta_records = [metric_row("full_oof", name, y, score) for name, score in delta_oof.items()]
    paired_am = df["final_alphamissense_score"].notna().to_numpy()
    delta_records.extend(metric_row("paired_alphamissense", name, y[paired_am], score[paired_am])
        for name, score in delta_oof.items())
    delta_metrics_df = pd.DataFrame(delta_records)
    print(delta_metrics_df.to_string(index=False))

    delta_out = df[["KEY", "Gene", "Mutation_used", "Mislocalized", "fold"]].copy()
    for name, score in delta_oof.items():
        delta_out[f"oof_{name}"] = score
    delta_out.to_csv(DELTA_OOF_CSV, index=False)
    delta_metrics_df.to_csv(DELTA_METRICS_CSV, index=False)
    if delta_importance_parts:
        delta_importance = pd.concat(delta_importance_parts).groupby(["model", "feature"], as_index=False)["importance"].mean()
        delta_importance["rank_within_model"] = delta_importance.groupby("model")["importance"].rank(method="min", ascending=False).astype(int)
        delta_importance.sort_values(["model", "rank_within_model"]).to_csv(DELTA_IMPORTANCE_CSV, index=False)
else:
    print("Delta features not available. Run the cell in section 18.7 first.")

               scope                      model    n  positives    auroc    auprc  precision_at_recall_0.20  precision_at_recall_0.40
            full_oof                 xgboost_64 2179        236 0.642168 0.198098                  0.266061                  0.186119
            full_oof               wt_signal_70 2179        236 0.656036 0.247897                  0.313681                  0.212524
            full_oof   xgboost_64_plus_delta_71 2179        236 0.631319 0.194914                  0.276349                  0.196912
            full_oof wt_signal_70_plus_delta_77 2179        236 0.643067 0.230235                  0.313887                  0.199938
paired_alphamissense                 xgboost_64 2140        235 0.644242 0.199925                  0.265537                  0.186508
paired_alphamissense               wt_signal_70 2140        235 0.657930 0.249871                  0.313333                  0.214123
paired_alphamissense   xgboost_64_plus_delta_71 2140        23

## 18.9 Paired gene-cluster bootstrap (delta models)

Comparison of the four delta-era models against xgboost_64 and against wt_signal_70.

In [11]:
if delta is not None:
    rng = np.random.default_rng(RANDOM_STATE)
    delta_bootstrap_rows = []
    for replicate in range(2000):
        sampled_genes = rng.choice(unique_genes, size=len(unique_genes), replace=True)
        idx = np.concatenate([gene_to_idx[gene] for gene in sampled_genes])
        if np.unique(y[idx]).size < 2:
            continue
        base_auc = roc_auc_score(y[idx], delta_oof["xgboost_64"][idx])
        base_ap = average_precision_score(y[idx], delta_oof["xgboost_64"][idx])
        signal_auc = roc_auc_score(y[idx], delta_oof["wt_signal_70"][idx])
        signal_ap = average_precision_score(y[idx], delta_oof["wt_signal_70"][idx])
        for name in ["wt_signal_70", "xgboost_64_plus_delta_71", "wt_signal_70_plus_delta_77"]:
            model_auc = roc_auc_score(y[idx], delta_oof[name][idx])
            model_ap = average_precision_score(y[idx], delta_oof[name][idx])
            delta_bootstrap_rows.append({
                "replicate": replicate,
                "comparison": f"{name}-xgboost_64",
                "delta_auroc": model_auc - base_auc,
                "delta_auprc": model_ap - base_ap})
            if name != "wt_signal_70":
                delta_bootstrap_rows.append({
                    "replicate": replicate,
                    "comparison": f"{name}-wt_signal_70",
                    "delta_auroc": model_auc - signal_auc,
                    "delta_auprc": model_ap - signal_ap})
    delta_bootstrap = pd.DataFrame(delta_bootstrap_rows)
    delta_bootstrap.to_csv(DELTA_BOOTSTRAP_CSV, index=False)
    delta_summary = delta_bootstrap.groupby("comparison").agg(
        delta_auroc_mean=("delta_auroc", "mean"),
        delta_auroc_low=("delta_auroc", lambda x: np.quantile(x, 0.025)),
        delta_auroc_high=("delta_auroc", lambda x: np.quantile(x, 0.975)),
        delta_auprc_mean=("delta_auprc", "mean"),
        delta_auprc_low=("delta_auprc", lambda x: np.quantile(x, 0.025)),
        delta_auprc_high=("delta_auprc", lambda x: np.quantile(x, 0.975))).reset_index()
    print(delta_summary.to_string(index=False))
    print("Do not claim superiority when the paired interval crosses zero.")
else:
    print("Delta features not available.")

                             comparison  delta_auroc_mean  delta_auroc_low  delta_auroc_high  delta_auprc_mean  delta_auprc_low  delta_auprc_high
                wt_signal_70-xgboost_64          0.013756        -0.019438          0.044861          0.047155         0.013017          0.084284
wt_signal_70_plus_delta_77-wt_signal_70         -0.013260        -0.033284          0.006625         -0.018049        -0.042075          0.004427
  wt_signal_70_plus_delta_77-xgboost_64          0.000496        -0.035562          0.034196          0.029106        -0.005266          0.064469
  xgboost_64_plus_delta_71-wt_signal_70         -0.024626        -0.052685          0.002414         -0.050987        -0.082407         -0.022549
    xgboost_64_plus_delta_71-xgboost_64         -0.010870        -0.033254          0.010923         -0.003832        -0.026989          0.018723
Do not claim superiority when the paired interval crosses zero.


## 18.9b Direct paired bootstrap: wt_signal_70 vs AlphaMissense

Gene-cluster bootstrap (2,000 replicates) on the 2,140-row paired AlphaMissense cohort. All three comparisons (wt_signal_70 vs AM, wt_signal_70 vs xgboost_64, xgboost_64 vs AM) use the same resampled genes within each replicate.

In [12]:
AM_BOOTSTRAP_CSV = BASE / "task18_alphamissense_bootstrap.csv"

paired_am = df["final_alphamissense_score"].notna().to_numpy()
y_am = df.loc[paired_am, "Mislocalized"].astype(int).to_numpy()
am_score = df.loc[paired_am, "final_alphamissense_score"].to_numpy()
wt70_am = oof["wt_signal_70"][paired_am]
xgb64_am = oof["xgboost_64"][paired_am]

genes_paired = df.loc[paired_am, "Gene"].astype(str).to_numpy()
unique_genes_am = np.unique(genes_paired)
gene_to_idx_am = {g: np.flatnonzero(genes_paired == g) for g in unique_genes_am}

print(f"Paired AM cohort: {len(y_am)} variants, {int(y_am.sum())} positives")
print(f"  AlphaMissense: AUROC={roc_auc_score(y_am, am_score):.4f}, AUPRC={average_precision_score(y_am, am_score):.4f}")
print(f"  wt_signal_70:   AUROC={roc_auc_score(y_am, wt70_am):.4f}, AUPRC={average_precision_score(y_am, wt70_am):.4f}")
print(f"  xgboost_64:     AUROC={roc_auc_score(y_am, xgb64_am):.4f}, AUPRC={average_precision_score(y_am, xgb64_am):.4f}")

rng = np.random.default_rng(RANDOM_STATE)
am_bootstrap_rows = []
for rep in range(2000):
    sampled = rng.choice(unique_genes_am, size=len(unique_genes_am), replace=True)
    idx = np.concatenate([gene_to_idx_am[g] for g in sampled])
    if np.unique(y_am[idx]).size < 2:
        continue
    am_auc = roc_auc_score(y_am[idx], am_score[idx])
    am_ap = average_precision_score(y_am[idx], am_score[idx])
    wt70_auc = roc_auc_score(y_am[idx], wt70_am[idx])
    wt70_ap = average_precision_score(y_am[idx], wt70_am[idx])
    xgb64_auc = roc_auc_score(y_am[idx], xgb64_am[idx])
    xgb64_ap = average_precision_score(y_am[idx], xgb64_am[idx])
    am_bootstrap_rows.append({
        "replicate": rep,
        "wt70-am_delta_auroc": wt70_auc - am_auc,
        "wt70-am_delta_auprc": wt70_ap - am_ap,
        "wt70-xgb64_delta_auroc": wt70_auc - xgb64_auc,
        "wt70-xgb64_delta_auprc": wt70_ap - xgb64_ap,
        "xgb64-am_delta_auroc": xgb64_auc - am_auc,
        "xgb64-am_delta_auprc": xgb64_ap - am_ap})

am_bootstrap = pd.DataFrame(am_bootstrap_rows)
am_bootstrap.to_csv(AM_BOOTSTRAP_CSV, index=False)

print("\n=== Paired gene-cluster bootstrap (2,000 replicates) ===")
for comp, label in [
    ("wt70-am", "wt_signal_70 \u2212 AlphaMissense"),
    ("wt70-xgb64", "wt_signal_70 \u2212 xgboost_64"),
    ("xgb64-am", "xgboost_64 \u2212 AlphaMissense"),
]:
    d_auc = am_bootstrap[f"{comp}_delta_auroc"]
    d_ap = am_bootstrap[f"{comp}_delta_auprc"]
    print(f"\n{label}:")
    print(f"  \u0394AUROC: {d_auc.mean():+.4f}  [95% CI {d_auc.quantile(0.025):+.4f}, {d_auc.quantile(0.975):+.4f}]")
    print(f"  \u0394AUPRC: {d_ap.mean():+.4f}  [95% CI {d_ap.quantile(0.025):+.4f}, {d_ap.quantile(0.975):+.4f}]")

print(f"\nSaved: {AM_BOOTSTRAP_CSV}")

Paired AM cohort: 2140 variants, 235 positives
  AlphaMissense: AUROC=0.6491, AUPRC=0.1619
  wt_signal_70:   AUROC=0.6579, AUPRC=0.2499
  xgboost_64:     AUROC=0.6442, AUPRC=0.1999

=== Paired gene-cluster bootstrap (2,000 replicates) ===

wt_signal_70 − AlphaMissense:
  ΔAUROC: +0.0088  [95% CI -0.0367, +0.0574]
  ΔAUPRC: +0.0866  [95% CI +0.0271, +0.1475]

wt_signal_70 − xgboost_64:
  ΔAUROC: +0.0138  [95% CI -0.0217, +0.0451]
  ΔAUPRC: +0.0477  [95% CI +0.0131, +0.0827]

xgboost_64 − AlphaMissense:
  ΔAUROC: -0.0050  [95% CI -0.0479, +0.0400]
  ΔAUPRC: +0.0389  [95% CI -0.0099, +0.0853]

Saved: /mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/task18_alphamissense_bootstrap.csv
